## Imports and constants

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_SEED = 42
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
SPLIT_DATE = "2026-08-01"        # train < this date, test >= this date
np.random.seed(RANDOM_SEED)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
WINDOW_START = pd.Timestamp("2026-01-01")
WINDOW_END   = pd.Timestamp("2026-11-01")   # exclusive upper bound
QUALITY_LOG  = []

def log_issue(table, issue, rows_found, rows_expected, handling_rule):
    """Every cleaning step calls this. rows_expected comes from
    chargebacklens_data_spec.md §3 — a mismatch means either the loader
    dropped something it shouldn't have, or the check itself is wrong."""
    QUALITY_LOG.append(dict(
        table=table, issue=issue,
        rows_found=int(rows_found), rows_expected=int(rows_expected),
        match=bool(int(rows_found) == int(rows_expected)),
        handling_rule=handling_rule))
    flag = "OK " if int(rows_found) == int(rows_expected) else "MISMATCH"
    print(f"[{flag}] {table:12s} {issue:35s} found={rows_found:>6} "
          f"expected={rows_expected:>6}  -> {handling_rule}")

## Load raw tables with explicit dtypes

In [6]:
DTYPES = {
    "transactions": {"payment_id": "string", "merchant_id": "string", "customer_id": "string",
                     "amount": "float64", "method": "string", "issuer_bank": "string",
                     "card_bin_country": "string", "ip_state": "string", "device_id": "string",
                     "is_first_txn_for_device": "boolean", "checkout_latency_ms": "int64",
                     "retry_count": "int64", "auth_status": "string"},
    "customers":    {"customer_id": "string", "account_age_days": "int64", "lifetime_orders": "int64",
                     "prior_disputes": "int64", "avg_order_value": "float64",
                     "email_domain_type": "string", "phone_verified": "boolean"},
    "merchants":    {"merchant_id": "string", "category": "string", "avg_ticket_size": "float64",
                     "refund_window_days": "int64", "delivery_sla_days": "Int64"},
    "fulfilment":   {"payment_id": "string", "delivery_status": "string",
                     "tracking_available": "boolean", "address_completeness_score": "float64"},
    "disputes":     {"dispute_id": "string", "payment_id": "string", "reason_code": "string",
                     "disputed_amount": "float64"},
}
DATE_COLS = {"transactions": ["created_at"], "customers": [], "merchants": [],
             "fulfilment": ["shipped_at", "delivered_at"], "disputes": ["raised_at"]}
EXPECTED_ROWS = {"transactions": 120000, "customers": 35000, "merchants": 60,
                 "fulfilment": 66140, "disputes": 1091}

def load_raw_tables(raw_dir: Path) -> dict[str, pd.DataFrame]:
    """Explicit dtype map per data spec §3 — no inference. Dates parsed at read time."""
    return {name: pd.read_csv(raw_dir / f"{name}.csv", dtype=dtype,
                              parse_dates=DATE_COLS[name])
            for name, dtype in DTYPES.items()}

tables = load_raw_tables(RAW_DIR)
for name, df in tables.items():
    print(f"{name:14s} {str(df.shape):>14}   expected {EXPECTED_ROWS[name]:>6}")
    assert df.shape[0] == EXPECTED_ROWS[name], f"{name}: got {df.shape[0]}"

transactions     (120000, 14)   expected 120000
customers          (35000, 7)   expected  35000
merchants             (60, 5)   expected     60
fulfilment         (66140, 6)   expected  66140
disputes            (1091, 5)   expected   1091


## Schema validation

In [8]:
EXPECTED_COLUMNS = {
    "transactions": {"payment_id","merchant_id","customer_id","amount","method","issuer_bank",
                     "card_bin_country","created_at","ip_state","device_id",
                     "is_first_txn_for_device","checkout_latency_ms","retry_count","auth_status"},
    "customers":    {"customer_id","account_age_days","lifetime_orders","prior_disputes",
                     "avg_order_value","email_domain_type","phone_verified"},
    "merchants":    {"merchant_id","category","avg_ticket_size","refund_window_days","delivery_sla_days"},
    "fulfilment":   {"payment_id","shipped_at","delivered_at","delivery_status",
                     "tracking_available","address_completeness_score"},
    "disputes":     {"dispute_id","payment_id","reason_code","raised_at","disputed_amount"},
}

def validate_schema(tables: dict[str, pd.DataFrame]) -> None:
    for name, expected in EXPECTED_COLUMNS.items():
        actual  = set(tables[name].columns)
        missing = expected - actual
        extra   = actual - expected
        assert not missing, f"{name}: missing columns {sorted(missing)}"
        if extra:
            print(f"WARNING {name}: extra columns {sorted(extra)} (ignored downstream)")
        print(f"schema OK: {name} ({len(expected)} columns)")

validate_schema(tables)

# Structural nulls (data spec §3.1) — assert the RULE, not a hardcoded count.
t = tables["transactions"]
assert t.loc[t["method"].isin(["upi","wallet"]), "issuer_bank"].isna().all()
assert t.loc[~t["method"].eq("card"), "card_bin_country"].isna().all()
assert (t["auth_status"] == "success").all()
print(f"structural nulls OK — issuer_bank {t['issuer_bank'].isna().sum()}, "
      f"card_bin_country {t['card_bin_country'].isna().sum()}")

schema OK: transactions (14 columns)
schema OK: customers (7 columns)
schema OK: merchants (5 columns)
schema OK: fulfilment (6 columns)
schema OK: disputes (5 columns)
structural nulls OK — issuer_bank 63600, card_bin_country 82800


## Foreign key integrity

In [9]:
txns       = tables["transactions"].copy()
customers  = tables["customers"].copy()
merchants  = tables["merchants"].copy()
fulfilment = tables["fulfilment"].copy()
disputes   = tables["disputes"].copy()

# First-party FKs — zero violations expected, so raise.
bad_merchant = (~txns["merchant_id"].isin(set(merchants["merchant_id"]))).sum()
bad_customer = (~txns["customer_id"].isin(set(customers["customer_id"]))).sum()
assert bad_merchant == 0, f"{bad_merchant} txns reference an unknown merchant_id"
assert bad_customer == 0, f"{bad_customer} txns reference an unknown customer_id"
print(f"FK txns->merchants: {bad_merchant} | txns->customers: {bad_customer}")

valid_pids = set(txns["payment_id"])

# Cross-system FKs — orphans are designed-in. Log, drop, assert the count.
ful_orphans = ~fulfilment["payment_id"].isin(valid_pids)
log_issue("fulfilment", "3.4.3 orphan payment_id", ful_orphans.sum(), 140, "dropped")
assert ful_orphans.sum() == 140
fulfilment = fulfilment.loc[~ful_orphans].copy()

dis_orphans = ~disputes["payment_id"].isin(valid_pids)
log_issue("disputes", "3.5.3 orphan payment_id", dis_orphans.sum(), 35, "dropped")
assert dis_orphans.sum() == 35
disputes = disputes.loc[~dis_orphans].copy()

FK txns->merchants: 0 | txns->customers: 0
[OK ] fulfilment   3.4.3 orphan payment_id             found=   140 expected=   140  -> dropped
[OK ] disputes     3.5.3 orphan payment_id             found=    35 expected=    35  -> dropped


## Duplicate-key resolution

In [11]:
# transactions: 12 payment_ids x2 -> keep the earlier created_at
txns = txns.sort_values(["payment_id", "created_at"], kind="mergesort")
n_before = len(txns)
txns = txns.drop_duplicates(subset="payment_id", keep="first").sort_index()
log_issue("transactions", "3.1.1 duplicate payment_id", n_before - len(txns), 12,
          "dropped later-created_at row")
assert txns["payment_id"].is_unique

# customers: 10 customer_ids x2 -> keep the higher lifetime_orders (more complete record)
customers = customers.sort_values(["customer_id", "lifetime_orders"],
                                  ascending=[True, False], kind="mergesort")
n_before = len(customers)
customers = customers.drop_duplicates(subset="customer_id", keep="first").sort_index()
log_issue("customers", "3.2.4 duplicate customer_id", n_before - len(customers), 10,
          "dropped lower-lifetime_orders row")
assert customers["customer_id"].is_unique

# disputes: 6 payment_ids x2 -> keep the earlier raised_at (the original filing)
disputes = disputes.sort_values(["payment_id", "raised_at"], kind="mergesort")
n_before = len(disputes)
disputes = disputes.drop_duplicates(subset="payment_id", keep="first").sort_index()
log_issue("disputes", "3.5.2 duplicate payment_id", n_before - len(disputes), 6,
          "dropped later-raised_at row")
assert disputes["payment_id"].is_unique and len(disputes) == 1050
print(f"label-defining disputes: {len(disputes)}")

[OK ] transactions 3.1.1 duplicate payment_id          found=    12 expected=    12  -> dropped later-created_at row
[OK ] customers    3.2.4 duplicate customer_id         found=    10 expected=    10  -> dropped lower-lifetime_orders row
[OK ] disputes     3.5.2 duplicate payment_id          found=     6 expected=     6  -> dropped later-raised_at row
label-defining disputes: 1050


## Value-level cleaning

In [12]:
# --- 3.1.2 ip_state: 110 distinct string variants -> 36 canonical codes ---
STATE_NAME_TO_CODE = {
    "ANDHRA PRADESH":"AP","ARUNACHAL PRADESH":"AR","ASSAM":"AS","BIHAR":"BR","CHANDIGARH":"CH",
    "CHHATTISGARH":"CG","DELHI":"DL","GOA":"GA","GUJARAT":"GJ","HARYANA":"HR",
    "HIMACHAL PRADESH":"HP","JAMMU AND KASHMIR":"JK","JHARKHAND":"JH","KARNATAKA":"KA",
    "KERALA":"KL","LADAKH":"LA","MADHYA PRADESH":"MP","MAHARASHTRA":"MH","MANIPUR":"MN",
    "MEGHALAYA":"ML","MIZORAM":"MZ","NAGALAND":"NL","ODISHA":"OD","PUDUCHERRY":"PY",
    "PUNJAB":"PB","RAJASTHAN":"RJ","SIKKIM":"SK","TAMIL NADU":"TN","TELANGANA":"TG",
    "TRIPURA":"TR","UTTAR PRADESH":"UP","UTTARAKHAND":"UK","WEST BENGAL":"WB",
}
raw_state  = txns["ip_state"].copy()
norm_state = raw_state.str.strip().str.upper().replace(STATE_NAME_TO_CODE)  # "GOA" needs the map too
log_issue("transactions", "3.1.2 non-standard ip_state", (norm_state != raw_state).sum(), 360,
          "normalised to 2-letter code")
txns["ip_state"] = norm_state
print(f"ip_state: {raw_state.nunique()} distinct -> {txns['ip_state'].nunique()}")

# --- 3.1.3 device_id: "" -> NA. Null the FIELD, never drop the ROW. ---
blank_device = txns["device_id"].isna() | txns["device_id"].str.strip().eq("")
log_issue("transactions", "3.1.3 empty device_id", blank_device.sum(), 60, "set to NA, row kept")
txns.loc[blank_device, "device_id"] = pd.NA

# --- 3.1.4 / 3.1.5 -> flag, don't drop (keeps master at 119,988) ---
bad_amount = txns["amount"] <= 0
log_issue("transactions", "3.1.4 non-positive amount", bad_amount.sum(), 8,
          "flagged exclude_from_modelling, row kept")
print("  payment_ids:", txns.loc[bad_amount, "payment_id"].tolist())

out_of_window = (txns["created_at"] < WINDOW_START) | (txns["created_at"] >= WINDOW_END)
log_issue("transactions", "3.1.5 out-of-window created_at", out_of_window.sum(), 3,
          "flagged exclude_from_modelling, row kept")
print("  dates:", txns.loc[out_of_window, "created_at"].dt.date.tolist())  # 2019-06-09, 2099-03-17, 2099-11-02

txns["exclude_from_modelling"] = (bad_amount | out_of_window).astype("int8")

# --- 3.1.6 clip, 3.1.7 cap-but-preserve ---
log_issue("transactions", "3.1.6 negative checkout_latency_ms",
          (txns["checkout_latency_ms"] < 0).sum(), 12, "clipped to 0")
txns["checkout_latency_ms"] = txns["checkout_latency_ms"].clip(lower=0)

RETRY_CAP = 10
log_issue("transactions", "3.1.7 implausible retry_count",
          (txns["retry_count"] > RETRY_CAP).sum(), 5, f"capped at {RETRY_CAP}, raw kept")
txns["retry_count_raw"] = txns["retry_count"]          # notebook 2 plots this, notebook 3 uses the cap
txns["retry_count"]     = txns["retry_count"].clip(upper=RETRY_CAP)

# --- customers ---
log_issue("customers", "3.2.3 negative account_age_days",
          (customers["account_age_days"] < 0).sum(), 6, "clipped to 0")
customers["account_age_days"] = customers["account_age_days"].clip(lower=0)

blank_email = customers["email_domain_type"].isna() | customers["email_domain_type"].str.strip().eq("")
log_issue("customers", "3.2.2 empty email_domain_type", blank_email.sum(), 15,
          "set to 'unknown' 4th category")
customers.loc[blank_email, "email_domain_type"] = "unknown"

# --- disputes ---
null_reason = disputes["reason_code"].isna() | disputes["reason_code"].str.strip().eq("")
log_issue("disputes", "3.5.4 missing reason_code", null_reason.sum(), 5, "set to 'unclassified'")
disputes.loc[null_reason, "reason_code"] = "unclassified"

# 3.5.5 — the dispute is real, only its timestamp is not. Label survives, timing field dies.
dt = disputes.merge(txns[["payment_id","created_at"]], on="payment_id",
                    how="left", validate="one_to_one")
impossible = (dt["raised_at"] < dt["created_at"]).to_numpy()
log_issue("disputes", "3.5.5 raised_at < created_at", impossible.sum(), 12,
          "raised_at nulled, dispute still labels")
disputes.loc[impossible, "raised_at"] = pd.NaT

# --- fulfilment: 3.4.2 MUST be measured before 3.4.1 mutates delivered_at ---
status_ts_mismatch = fulfilment["delivery_status"].eq("delivered") & fulfilment["delivered_at"].isna()
log_issue("fulfilment", "3.4.2 delivered but no delivered_at", status_ts_mismatch.sum(), 330,
          "flagged only, not repaired")
fulfilment["delivered_ts_missing"] = status_ts_mismatch.astype("int8")

bad_seq = fulfilment["delivered_at"] < fulfilment["shipped_at"]
log_issue("fulfilment", "3.4.1 delivered_at < shipped_at", bad_seq.sum(), 790,
          "both timestamps nulled")
fulfilment.loc[bad_seq, ["shipped_at","delivered_at"]] = pd.NaT
fulfilment["timestamps_untrusted"] = bad_seq.astype("int8")

log_issue("fulfilment", "3.4.4 address score out of [0,1]",
          ((fulfilment["address_completeness_score"] < 0) |
           (fulfilment["address_completeness_score"] > 1)).sum(), 25, "clipped to [0,1]")
fulfilment["address_completeness_score"] = fulfilment["address_completeness_score"].clip(0.0, 1.0)

# --- merchants: 3.3.1 is the reason is_physical_goods exists at all ---
sla_on_digital = (merchants["category"].isin(["gaming","edtech","subscription"])
                  & merchants["delivery_sla_days"].notna())
log_issue("merchants", "3.3.1 SLA on non-physical category", sla_on_digital.sum(), 2,
          "left as-is, flagged")
merchants["is_physical_goods"] = merchants["delivery_sla_days"].notna().astype("int8")
log_issue("merchants", "3.3.2 refund_window_days == 0",
          (merchants["refund_window_days"] == 0).sum(), 1, "left as-is, noted")

[OK ] transactions 3.1.2 non-standard ip_state         found=   360 expected=   360  -> normalised to 2-letter code
ip_state: 110 distinct -> 36
[OK ] transactions 3.1.3 empty device_id               found=    60 expected=    60  -> set to NA, row kept
[OK ] transactions 3.1.4 non-positive amount           found=     8 expected=     8  -> flagged exclude_from_modelling, row kept
  payment_ids: ['pay_1cx3pio46clgxx', 'pay_xp71axkq637rs4', 'pay_nsvv23y8mgtzuy', 'pay_n37jk9ssebbtzb', 'pay_6t0q77u70t40fn', 'pay_muu8z9hu6hmkhv', 'pay_z1cprfy3bcp8ui', 'pay_g67s1z01zt8f5m']
[OK ] transactions 3.1.5 out-of-window created_at      found=     3 expected=     3  -> flagged exclude_from_modelling, row kept
  dates: [datetime.date(2019, 6, 9), datetime.date(2099, 3, 17), datetime.date(2099, 11, 2)]
[OK ] transactions 3.1.6 negative checkout_latency_ms  found=    12 expected=    12  -> clipped to 0
[OK ] transactions 3.1.7 implausible retry_count       found=     5 expected=     5  -> capped at 10, r

## Build the label

In [13]:
def build_label(transactions: pd.DataFrame, disputes: pd.DataFrame) -> pd.DataFrame:
    """Left-join on payment_id. Adds is_disputed (int8), dispute_raised_at,
    dispute_reason_code. The uniqueness asserts are the safety net for Cell 5's
    dedup — not the primary fix."""
    d = disputes[["payment_id","raised_at","reason_code"]].rename(
        columns={"raised_at":"dispute_raised_at", "reason_code":"dispute_reason_code"})
    assert d["payment_id"].is_unique, "disputes still has duplicate payment_id at label time"

    out = transactions.merge(d, on="payment_id", how="left", validate="one_to_one")
    assert len(out) == len(transactions), "label join changed the row count"
    assert out["payment_id"].is_unique, "payment_id duplicated after label join"

    out["is_disputed"] = (out["dispute_raised_at"].notna()
                          | out["dispute_reason_code"].notna()).astype("int8")
    return out

labelled  = build_label(txns, disputes)
n_pos     = int(labelled["is_disputed"].sum())
base_rate = n_pos / len(labelled)
print(f"positives: {n_pos} / {len(labelled)} = {base_rate:.4%}")
assert n_pos == 1050
assert 0.008 <= base_rate <= 0.010, "base rate outside data spec range — STOP and debug"

positives: 1050 / 119988 = 0.8751%


## Merge static merchant attributes

In [15]:
MERCHANT_COLS = ["merchant_id","category","avg_ticket_size","refund_window_days",
                 "delivery_sla_days","is_physical_goods"]

master = labelled.merge(
    merchants[MERCHANT_COLS].rename(columns={"category":"merchant_category"}),
    on="merchant_id", how="left", validate="many_to_one")

assert len(master) == len(labelled)
assert master["merchant_category"].notna().all()

# Permanent leakage tripwire (dev plan §2.2). prior_disputes is in here deliberately:
# the stale snapshot column must never reach master — notebook 3 recomputes it time-gated.
FORBIDDEN = ["delivered_at","delivery_status","shipped_at","tracking_available",
             "address_completeness_score","prior_disputes"]
leaked = [c for c in FORBIDDEN if c in master.columns]
assert not leaked, f"leakage tripwire fired: {leaked}"
print(f"01_master_labelled: {master.shape}  |  tripwire clean")

01_master_labelled: (119988, 24)  |  tripwire clean


## Data quality log

In [16]:
quality_log = pd.DataFrame(QUALITY_LOG)[
    ["table","issue","rows_found","rows_expected","match","handling_rule"]]
print(quality_log.to_string(index=False))
print(f"\nFALSE rows: {int((~quality_log['match']).sum())}")
assert quality_log["match"].all(), "quality log has a mismatch — do not proceed to notebook 2"

       table                               issue  rows_found  rows_expected  match                            handling_rule
  fulfilment             3.4.3 orphan payment_id         140            140   True                                  dropped
    disputes             3.5.3 orphan payment_id          35             35   True                                  dropped
transactions          3.1.1 duplicate payment_id          12             12   True             dropped later-created_at row
   customers         3.2.4 duplicate customer_id          10             10   True        dropped lower-lifetime_orders row
    disputes          3.5.2 duplicate payment_id           6              6   True              dropped later-raised_at row
transactions         3.1.2 non-standard ip_state         360            360   True              normalised to 2-letter code
transactions               3.1.3 empty device_id          60             60   True                      set to NA, row kept
transact

## Export with round-trip verification

In [17]:
def export(df: pd.DataFrame, filename: str) -> None:
    path = PROCESSED_DIR / filename
    df.to_csv(path, index=False)
    check = pd.read_csv(path)
    assert check.shape[0] == df.shape[0], \
        f"{filename}: wrote {df.shape[0]}, read {check.shape[0]}"
    assert set(check.columns) == set(df.columns), f"{filename}: column mismatch after round-trip"
    print(f"✅ {filename} — {check.shape[0]} rows, {check.shape[1]} cols")

export(txns,        "01_cleaned_transactions.csv")
export(customers,   "01_cleaned_customers.csv")
export(merchants,   "01_cleaned_merchants.csv")
export(fulfilment,  "01_cleaned_fulfilment.csv")
export(disputes,    "01_cleaned_disputes.csv")
export(master,      "01_master_labelled.csv")
export(quality_log, "01_data_quality_log.csv")

assert len(master) == 119_988, "master row count broke the definition-of-done"

✅ 01_cleaned_transactions.csv — 119988 rows, 16 cols
✅ 01_cleaned_customers.csv — 34990 rows, 7 cols
✅ 01_cleaned_merchants.csv — 60 rows, 6 cols
✅ 01_cleaned_fulfilment.csv — 66000 rows, 8 cols
✅ 01_cleaned_disputes.csv — 1050 rows, 5 cols
✅ 01_master_labelled.csv — 119988 rows, 24 cols
✅ 01_data_quality_log.csv — 20 rows, 6 cols
